# Notebook 2: Accessing Materials Data — Materials Project

**APS Tutorial T4 — Generative AI for Physics: From Models to Materials**

In this notebook you will learn to:
- Query the **Materials Project** database using the `mp-api` Python client
- Search for materials by formula, elements, and properties
- Download and inspect crystal structures
- Understand SCIGEN's training data and why we need generative models

> **Prerequisites:** Run `00_setup.ipynb` first. An MP API key is optional — we provide fallback data.

In [ ]:
# --- Run once per Colab session (skip if you already ran 00_setup.ipynb) ---
import os, shutil, subprocess, sys

REPO_URL = 'https://github.com/RyotaroOKabe/APS_demo_SCIGEN.git'
PROJECT_DIR = '/content/APS_demo_SCIGEN'

# Clone repo if needed
if not os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    os.system(f'git clone {REPO_URL} {PROJECT_DIR}')

# Install all tutorial dependencies from the shared requirements file
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       '-r', f'{PROJECT_DIR}/requirements-colab.txt'])
os.environ.setdefault('PROJECT_ROOT', PROJECT_DIR)
print('Ready.')

---
## 2.1 The Materials Project

The [Materials Project](https://materialsproject.org) is the largest open database of
computed materials properties, with ~150,000 inorganic crystal structures.

Each entry includes:
- Crystal structure (atomic positions, lattice)
- Formation energy, band gap, energy above hull
- Magnetic properties, elastic properties, and more

SCIGEN was trained on the **MP-20** subset: structures with ≤20 atoms per unit cell.

---
## 2.2 Setting up the MP API client

To query the Materials Project live, you need a free API key from
[materialsproject.org/api](https://materialsproject.org/api).

If you don't have one, skip to **Section 2.4** where we use pre-downloaded data.

In [ ]:
# Set your API key here (or leave as None to use fallback data)
MP_API_KEY = None  # e.g., 'your_api_key_here'

USE_MP_API = MP_API_KEY is not None

if USE_MP_API:
    from mp_api.client import MPRester
    mpr = MPRester(MP_API_KEY)
    print('MP API client ready.')
else:
    print('No API key provided — using pre-downloaded data.')
    print('(To enable live queries, get a free key at materialsproject.org/api)')

---
## 2.3 Querying the Materials Project

Let's search for manganese-containing materials — the kind of materials
we'll later generate with SCIGEN.

In [ ]:
if USE_MP_API:
    # Search for stable Mn-containing materials
    results = mpr.materials.summary.search(
        elements=['Mn'],
        energy_above_hull=(0, 0.05),  # near-stable materials
        num_sites=(1, 20),  # small unit cells (like SCIGEN's training data)
        fields=['material_id', 'formula_pretty', 'formation_energy_per_atom',
                'energy_above_hull', 'nsites', 'symmetry']
    )
    print(f'Found {len(results)} near-stable Mn-containing materials with ≤20 atoms/cell')
    print()
    for r in results[:10]:
        print(f'  {r.material_id}  {r.formula_pretty:<15s}  '
              f'Ef={r.formation_energy_per_atom:.3f} eV/atom  '
              f'Ehull={r.energy_above_hull:.3f} eV  '
              f'nsites={r.nsites}')
else:
    print('(Skipping live query — no API key)')

In [ ]:
if USE_MP_API:
    # Download a specific structure
    struct_mp = mpr.get_structure_by_material_id('mp-35')  # Mn metal
    print(f'Downloaded: Mn metal (mp-35)')
    print(struct_mp)
else:
    print('(Skipping live download — no API key)')

---
## 2.4 Exploring SCIGEN's training data

Even without an API key, we can explore the MP-20 dataset that SCIGEN was trained on.
This data is included in the repository.

In [ ]:
import pandas as pd
import os

PROJECT_DIR = os.environ.get('PROJECT_ROOT', '/content/APS_demo_SCIGEN')

train_df = pd.read_csv(os.path.join(PROJECT_DIR, 'data', 'mp_20', 'train.csv'))
val_df = pd.read_csv(os.path.join(PROJECT_DIR, 'data', 'mp_20', 'val.csv'))
test_df = pd.read_csv(os.path.join(PROJECT_DIR, 'data', 'mp_20', 'test.csv'))

print(f'MP-20 dataset:')
print(f'  Train: {len(train_df):,} structures')
print(f'  Val:   {len(val_df):,} structures')
print(f'  Test:  {len(test_df):,} structures')
print(f'  Total: {len(train_df) + len(val_df) + len(test_df):,} structures')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Formation energy distribution
axes[0].hist(train_df['formation_energy_per_atom'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Formation energy (eV/atom)')
axes[0].set_ylabel('Count')
axes[0].set_title('Formation energy distribution')

# Band gap distribution
axes[1].hist(train_df['band_gap'], bins=60, color='coral', edgecolor='white')
axes[1].set_xlabel('Band gap (eV)')
axes[1].set_ylabel('Count')
axes[1].set_title('Band gap distribution')

# Energy above hull
axes[2].hist(train_df['e_above_hull'], bins=60, color='seagreen', edgecolor='white',
             range=(0, 0.5))
axes[2].set_xlabel('Energy above hull (eV/atom)')
axes[2].set_ylabel('Count')
axes[2].set_title('Stability (e_above_hull)')

plt.tight_layout()
plt.show()

In [ ]:
# Periodic table heatmap of element frequency in training data
from collections import Counter
import ast
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

all_elements = []
for elem_str in train_df['elements']:
    try:
        elems = ast.literal_eval(elem_str)
        all_elements.extend(elems)
    except:
        pass

elem_counts = Counter(all_elements)

# Periodic table layout (row, col) for each element
pt_layout = {
    'H': (0,0), 'He': (0,17),
    'Li': (1,0), 'Be': (1,1), 'B': (1,12), 'C': (1,13), 'N': (1,14), 'O': (1,15), 'F': (1,16), 'Ne': (1,17),
    'Na': (2,0), 'Mg': (2,1), 'Al': (2,12), 'Si': (2,13), 'P': (2,14), 'S': (2,15), 'Cl': (2,16), 'Ar': (2,17),
    'K': (3,0), 'Ca': (3,1), 'Sc': (3,2), 'Ti': (3,3), 'V': (3,4), 'Cr': (3,5), 'Mn': (3,6), 'Fe': (3,7),
    'Co': (3,8), 'Ni': (3,9), 'Cu': (3,10), 'Zn': (3,11), 'Ga': (3,12), 'Ge': (3,13), 'As': (3,14),
    'Se': (3,15), 'Br': (3,16), 'Kr': (3,17),
    'Rb': (4,0), 'Sr': (4,1), 'Y': (4,2), 'Zr': (4,3), 'Nb': (4,4), 'Mo': (4,5), 'Tc': (4,6), 'Ru': (4,7),
    'Rh': (4,8), 'Pd': (4,9), 'Ag': (4,10), 'Cd': (4,11), 'In': (4,12), 'Sn': (4,13), 'Sb': (4,14),
    'Te': (4,15), 'I': (4,16), 'Xe': (4,17),
    'Cs': (5,0), 'Ba': (5,1), 'La': (5,2), 'Hf': (5,3), 'Ta': (5,4), 'W': (5,5), 'Re': (5,6), 'Os': (5,7),
    'Ir': (5,8), 'Pt': (5,9), 'Au': (5,10), 'Hg': (5,11), 'Tl': (5,12), 'Pb': (5,13), 'Bi': (5,14),
}

fig, ax = plt.subplots(figsize=(14, 6))
max_count = max(elem_counts.values()) if elem_counts else 1
cmap = plt.cm.YlOrRd

for el, (row, col) in pt_layout.items():
    count = elem_counts.get(el, 0)
    color = cmap(count / max_count) if count > 0 else '#f0f0f0'
    rect = mpatches.FancyBboxPatch((col, -row), 0.9, 0.9, boxstyle='round,pad=0.05',
                                     facecolor=color, edgecolor='gray', linewidth=0.5)
    ax.add_patch(rect)
    ax.text(col + 0.45, -row + 0.55, el, ha='center', va='center', fontsize=8, fontweight='bold')
    if count > 0:
        ax.text(col + 0.45, -row + 0.2, str(count), ha='center', va='center', fontsize=6, color='gray')

ax.set_xlim(-0.5, 18.5)
ax.set_ylim(-6.5, 1.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Element frequency in MP-20 training data', fontsize=14, pad=15)

# Colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, max_count))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.5, aspect=20, pad=0.02)
cbar.set_label('Number of structures', fontsize=10)

plt.tight_layout()
plt.show()

### Spacegroup distribution

Crystal structures are classified by their **space group** — the set of symmetry operations
(rotations, reflections, translations) that leave the structure unchanged. There are 230 possible
space groups. Let's see which ones dominate the training data:

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
sg_counts = train_df['spacegroup.number'].value_counts().head(15)
ax.bar(range(len(sg_counts)), sg_counts.values, color='steelblue', edgecolor='white')
ax.set_xticks(range(len(sg_counts)))
ax.set_xticklabels([f'SG {sg}' for sg in sg_counts.index], rotation=45, ha='right')
ax.set_ylabel('Count')
ax.set_title('Top 15 space groups in MP-20 training data')
plt.tight_layout()
plt.show()

### Unit cell sizes

SCIGEN is trained on structures with ≤20 atoms per unit cell (the "MP-20" subset).
Here's the distribution:

---
## Exercises

1. How many structures in the training data have formation energy below -2 eV/atom? What elements dominate those structures?
2. Find the structure with the largest band gap. What is its formula?

In [ ]:
# Your code here

In [ ]:
# Count atoms per structure from CIF data
from pymatgen.core import Structure

n_atoms_list = []
for i in range(min(500, len(train_df))):  # sample for speed
    try:
        s = Structure.from_str(train_df.iloc[i]['cif'], fmt='cif')
        n_atoms_list.append(s.num_sites)
    except:
        pass

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(n_atoms_list, bins=range(1, 22), color='steelblue', edgecolor='white', align='left')
ax.set_xlabel('Atoms per unit cell')
ax.set_ylabel('Count (sample of 500)')
ax.set_title('Unit cell size distribution in MP-20')
ax.set_xticks(range(1, 21))
plt.tight_layout()
plt.show()

---
## 2.5 How many kagome materials exist?

This is the motivating question for SCIGEN: if we search the known databases,
how many materials with a kagome sublattice do we find?

The answer is: **very few.** Kagome magnets are rare in nature, but they are
extremely interesting for physics (frustrated magnetism, spin liquids, flat bands).

SCIGEN's goal is to **generate new candidate materials** with these special lattice
geometries — expanding the search space far beyond what's been experimentally observed.

In [ ]:
# Count Mn-containing materials in the training data
mn_count = 0
for elem_str in train_df['elements']:
    try:
        if 'Mn' in ast.literal_eval(elem_str):
            mn_count += 1
    except:
        pass

print(f'Mn-containing structures in training data: {mn_count} / {len(train_df)}')
print(f'That\'s {mn_count/len(train_df)*100:.1f}% of the dataset.')
print()
print('Of these, even fewer have a kagome sublattice.')
print('This is exactly why we need generative models —')
print('to explore regions of materials space that are underrepresented in databases.')

---
## Key takeaways

1. **Materials databases are large** (~150K structures in MP) but they mostly cover
   well-known, thermodynamically stable materials.
2. **Exotic geometries are rare** — kagome, honeycomb, and other frustrated lattices
   appear in only a handful of known structures.
3. **Generative models** can help explore the vast space of *possible* materials
   that haven't been made yet.

In the next notebook, we'll learn how these generative models work.

---
## What's next?

Proceed to **Notebook 03: Generative AI Concepts** to learn how diffusion models generate crystal structures.